# Language model and sequence generation

In this demo, we will study the different ways of decoding text from a trained model. This step is needed for many natural language processing tasks: translation, question answering, generation…

We will take the example of text generation. For a long time, it relied on more or less sophisticated fill-in-the-blank texts. GPT-2 (2019) showed that a neural network trained on large amounts of text writes fluent paragraphs. It is a *decoder-only* transformer: today's large language models (GPT-4, Llama, Mistral…) rely on the same principle and produce their answers with the decoding methods shown here.

We will start from the simplest method and work our way up to the methods used in practice, writing the choice of the next token ourselves. This shows the role of the softmax at the output of language models. To finish, we will find all these settings in KerasHub's `generate` method.

An [excellent article](https://huggingface.co/blog/how-to-generate) by Hugging Face can serve as a complementary resource to this notebook.

This demo doesn't need a GPU: GPT-2 small runs on CPU, where each test on the 7 prompts takes about a minute. It uses the GPU if there is one.

**Warning**: these models were trained on web text, without filtering. The generated texts, especially by sampling, can be incoherent, false or offensive.

## Installation

The [KerasHub](https://keras.io/keras_hub/) library provides pretrained models for Keras, GPT-2 among them. It is preinstalled on Colab; the following cell installs it elsewhere (Codespace, personal computer).

In [1]:
!pip install -q keras-hub

## Choice of language

We will see how to generate text in English but also in French, by loading models pretrained on different corpora. The following cell changes the language used for the rest of the demo.

In [2]:
lang = "en"
# lang = "fr"

## Loading a trained model

A KerasHub model is loaded with its class's `from_preset` method, `keras_hub.models.GPT2CausalLM` for a GPT-2 that predicts the next token (*causal language model*): it fetches the architecture, the weights and the tokenizer.

- In English, `"gpt2_base_en"` is OpenAI's GPT-2 small (2019), published by KerasHub on [Kaggle](https://www.kaggle.com/models/keras/gpt2).
- In French, [BelGPT-2](https://huggingface.co/antoinelouis/belgpt2) is a GPT-2 small trained on a French corpus. KerasHub loads it from the Hugging Face Hub with the `hf://` prefix. Its vocabulary names `</s>` the end of text token that KerasHub expects under the name `<|endoftext|>`: the cell therefore builds the tokenizer itself, after renaming that token.

The model's tokenizer is then found in `causal_lm.preprocessor.tokenizer`.

In [3]:
import functools
import json
import statistics
import warnings

import keras
import keras_hub
import tensorflow as tf
import tqdm.auto
from huggingface_hub import hf_hub_download

# Same random draws at each run of the notebook
keras.utils.set_random_seed(0)

if lang == "fr":
  repo = "antoinelouis/belgpt2"
  with open(hf_hub_download(repo, "vocab.json"), encoding="utf-8") as file:
    vocabulary = json.load(file)
  # The end of text token, under the name KerasHub expects
  vocabulary["<|endoftext|>"] = vocabulary.pop("</s>")
  tokenizer = keras_hub.models.GPT2Tokenizer(
    vocabulary, hf_hub_download(repo, "merges.txt")
  )
  causal_lm = keras_hub.models.GPT2CausalLM.from_preset(
    f"hf://{repo}",
    preprocessor=keras_hub.models.GPT2CausalLMPreprocessor(tokenizer),
  )
else:
  causal_lm = keras_hub.models.GPT2CausalLM.from_preset("gpt2_base_en")

tokenizer = causal_lm.preprocessor.tokenizer

## Inspecting the model

After loading a pretrained model, it is always a good idea to check its architecture and characteristics. The following cell displays the summary of the model, that of its *backbone* (the transformer, without the output layer), the configuration of the backbone and that of its first block. The next one extracts the useful figures.

In [4]:
causal_lm.summary()
causal_lm.backbone.summary()
print(causal_lm.backbone.get_config())
print(causal_lm.backbone.get_layer("transformer_layer_0").get_config())

Preprocessor: "gpt2_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gpt2_tokenizer (GPT2Tokenizer)                                │                       Vocab size: 50,257 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gpt2_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gpt2_backbone (GPT2Backbone)  │ (None, None, 768)         │     124,439,808 │ padding_mask[0][0],        │
│                               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 50257)       │      38,597,376 │ gpt2_backbone[0][0]        │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 124,439,808 (474.70 MB)

 Trainable params: 124,439,808 (474.70 MB)

... (3 outputs left out)

 Total params: 124,439,808 (474.70 MB)

 Trainable params: 124,439,808 (474.70 MB)

 Non-trainable params: 0 (0.00 B)

{'name': 'gpt2_backbone', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'vocabulary_size': 50257, 'num_layers': 12, 'num_heads': 12, 'hidden_dim': 768, 'intermediate_dim': 3072, 'dropout': 0.1, 'max_sequence_length': 1024}
{'name': 'transformer_layer_0', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'intermediate_dim': 3072, 'num_heads': 12, 'dropout': 0.1, 'activation': {'module': 'builtins', 'class_name': 'function', 'config': 'keras_hub>gelu_approximate', 'registered_name': 'function'}, 'layer_norm_epsilon': 1e-05, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'RandomNormal', 'config': {'seed': None, 'mean': 0.0, 'stddev': 0.02}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'normalize_first':

In [5]:
config = causal_lm.backbone.get_config()
print(f"Transformer layers: {config['num_layers']}")
print(f"Dropout: {config['dropout']}")
print(f"{causal_lm.count_params() / 1e6:.0f} million parameters")
print(f"Beginning/end of text: {tokenizer.start_token_id} / {tokenizer.end_token_id}")

Transformer layers: 12
Dropout: 0.1
124 million parameters
Beginning/end of text: 50256 / 50256


- The model has 12 transformer layers (`num_layers`, or the blocks `transformer_layer_0` to `transformer_layer_11` in the summary).
- Its regularization mechanism is dropout, with a rate of 0.1 (`dropout`, the `embeddings_dropout` layer and the dropout of each block).
- Its training stabilization mechanisms are layer normalization (the output `layer_norm` layer, and in each block `layer_norm_epsilon` and `normalize_first`: a normalization before the attention and before the dense layer) and the residual connections around the attention and the dense layer of each block (internal to `TransformerDecoder`, they don't show in the summary).
- It has about 124 million parameters. The output layer `token_embedding (ReversibleEmbedding)` adds none: it reuses the input's token embedding matrix.
- `tokenizer.start_token_id` and `tokenizer.end_token_id` give the index of the beginning and end of text token: 50256 (`<|endoftext|>`) for GPT-2, 2 for BelGPT-2. We will need it to encode the texts.

## The generation mechanism

To generate text from a trained model, we proceed iteratively: we generate the tokens one by one, starting from an empty text or from a sentence we want to complete.

To generate a token, we feed the network the tokens generated so far (as indices in the vocabulary). The network then outputs one score per vocabulary token. We pick one of these tokens, append it to what has already been generated, and start again to generate the next token.

More formally, we try to maximize $P(\text{Text}| \text{Initial})$ with the following decomposition:

$$P(\text{Text}| \text{Initial}) = \prod_{i=1}^{\text{Text length}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

The whole point of decoding is to find the text that maximizes this product: it is the model's best proposal. This value can't be computed exactly, so we use heuristics to approximate it.

The transformer's **causal mask** is what makes this decomposition possible: in each attention layer, position $i$ can only look at positions $j \le i$. During training, the model thus learns to predict each token from the previous tokens only: exactly what generation needs.

## Tokenization

The tokenizer loaded with the model turns a sentence into vocabulary indices (*subwords*) the model understands. In the display of the tokens, `Ġ` marks a token that starts with a space.

In the rest of the demo, we use the following `encode` and `decode` functions. `encode` adds the beginning of text token, and returns a tensor of shape `(1, number of tokens)`.

BelGPT-2 was trained on texts where punctuation is separated from words by spaces ("j' ai", "fin ."): its generations keep a trace of it. It also sometimes glues two sentences together without a space ("soir.Les"): a flaw of the model (line breaks removed from its corpus, most likely), not of the library.

In [6]:
def encode(sentence: str):
  """Token indices, after the beginning token: shape (1, number of tokens)."""
  token_ids = keras.ops.convert_to_tensor(tokenizer(sentence), dtype="int32")
  start = keras.ops.convert_to_tensor([tokenizer.start_token_id], dtype="int32")
  return keras.ops.expand_dims(keras.ops.concatenate([start, token_ids]), axis=0)


def decode(token_ids) -> str:
  """Text of the tokens, without the beginning token."""
  return tokenizer.detokenize(token_ids[0, 1:])


example = encode(
  "Je pense, donc je suis" if lang == "fr" else "I think, therefore I am"
)
print(example)
print([tokenizer.id_to_token(int(i)) for i in keras.ops.convert_to_numpy(example[0])])

tf.Tensor([[50256    40   892    11  4361   314   716]], shape=(1, 7), dtype=int32)
['<|endoftext|>', 'I', 'Ġthink', ',', 'Ġtherefore', 'ĠI', 'Ġam']


## Studying the outputs of the pretrained model

The first step to decode sentences from a trained model is to run a forward pass of the model to get its predictions for the next token. Let's study its behavior.

Like any Keras model, `causal_lm` is called like a function. It takes a dictionary of two tensors of shape `(batch, number of tokens)`:

- `"token_ids"`: the token indices;
- `"padding_mask"`: 1 for a real token, 0 for a padding position (here, we only have 1s, which [`keras.ops.ones_like`](https://keras.io/api/ops/numpy/#oneslike-function) gives).

It returns the **logits**: the model's scores, before softmax. Let's look at their shape for the empty text and for "I think, therefore I am".

In [7]:
# The empty text has a single position: Keras would warn that the attention's
# softmax is over an axis of size 1, which is expected here
warnings.filterwarnings("ignore", message="You are using a softmax over axis")


def describe_shapes(sentence: str) -> None:
  token_ids = encode(sentence)
  logits = causal_lm(
    {"token_ids": token_ids, "padding_mask": keras.ops.ones_like(token_ids)}
  )
  print(f"Shape of the logits for '{sentence}': {tuple(logits.shape)}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")

Shape of the logits for '': (1, 1, 50257)
Shape of the logits for 'I think, therefore I am': (1, 7, 50257)


The logits have the shape `(batch, number of tokens, vocabulary size)`: for each position of the input, the model gives a score to each of the 50,257 tokens of the vocabulary for the **next** position. The empty text has one position, that of the beginning token `encode` adds. To generate, only the last position matters.

## A `forward` function suited to decoding

We will use the following `forward` function, which returns a result suited to decoding: only the last position matters (the prediction of the next token).

It uses KerasHub's cache (the model's `call_with_cache` method): it keeps in memory the attention keys and values of the tokens already seen, which avoids recomputing them at each new token. When the cache is given, only **the new tokens** must be passed to the model. The cache has a fixed size, `CACHE_SIZE` positions, and `tf.function` compiles the model's step into a TensorFlow graph: the first call takes a few seconds, the next ones are fast.

In [8]:
CACHE_SIZE = 128  # positions the cache can hold: prompt and generated tokens


@tf.function
def cached_call(token_ids, cache, index):
  """One step of the model on one token, with the cache, compiled by TensorFlow."""
  logits, _, cache = causal_lm.call_with_cache(token_ids, cache, index)
  return logits[:, -1, :], cache


def forward(token_ids, past=None):
  """Logits of the next token, of shape (1, vocabulary size), and the cache."""
  if past is None:
    backbone = causal_lm.backbone
    head_dim = backbone.hidden_dim // backbone.num_heads
    shape = (1, backbone.num_layers, 2, CACHE_SIZE, backbone.num_heads, head_dim)
    past = (keras.ops.zeros(shape), 0)
  cache, seen = past
  # One token at a time: the compiled function always sees the same shapes
  for position in range(token_ids.shape[1]):
    logits, cache = cached_call(
      token_ids[:, position : position + 1], cache, tf.constant(seen + position)
    )
  return logits, (cache, seen + token_ids.shape[1])

So use either:

- `forward(all_tokens)`
- `forward(last_token, past)`

where `all_tokens` would be for example a sequence of 7 tokens of shape `(1, 7)` while `last_token` would be of shape `(1, 1)`. The logits returned have the shape `(1, vocabulary size)`: the batch dimension, of size 1, is kept.

The model still computes logits for every position: they are used in training, where each position predicts the token that follows it, all at once in a single pass. The causal mask prevents position $i$ from seeing the next tokens, otherwise it would directly read the token it has to predict.

## Decoding loop

A decoding loop always follows the same pattern:

1. Forward pass on what has been produced so far (inputs and previous outputs; thanks to the cache, only the last token after the first step)
2. Choice of the index of the next token
3. Repetition of 1. and 2. until a stopping criterion is met (in this demo, only the number of tokens produced)

Step 2 is where everything happens. The following code implements all the rest, so that we can focus on step 2: each decoding method will be a function that receives the logits of the last token (shape `(1, vocabulary size)`) and returns the index of the chosen token (a tensor of shape `()`). Decorated with `@decoding_loop`, it becomes a function `(prompt, number of tokens to generate, parameters of the method)` that returns the generated text.

In [9]:
def decoding_loop(step_function):

  @functools.wraps(step_function)
  def wrapper(prompt: str, length: int, *step_args, **step_kwargs) -> str:
    decoded = [encode(prompt)]
    past = None
    for _ in tqdm.auto.trange(length, desc="Tokens", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded.append(keras.ops.cast(keras.ops.reshape(index, (1, 1)), "int32"))
    return decode(keras.ops.concatenate(decoded, axis=1))

  return wrapper

## Greedy decoding

Greedy decoding is the simplest method to decode text from a trained model: at each step, we take the token with the highest probability.

In the equation:

$$P(\text{Text}|\text{Initial}) = \prod_{i=1}^{\text{Text length}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

greedy decoding chooses $\text{Token}_i$ to maximize $P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$. However, it is often necessary to choose a less probable token to then reach very probable tokens.

The following example comes from Hugging Face's article on text generation:

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

After "The", we choose "nice" because it is the most probable word, whereas the complete text "The nice woman" has a probability (0.20) lower than the one we would have obtained by choosing "dog" ("The dog has", 0.36).

The `greedy` function returns the index of the largest logit, with [`keras.ops.argmax`](https://keras.io/api/ops/numpy/#argmax-function): no need for the softmax, which doesn't change the order of the scores. Let's generate 10 tokens after "I went to the".

In [10]:
@decoding_loop
def greedy(logits):
  return keras.ops.argmax(logits, axis=-1)[0]


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))

I went to the store and bought a new pair of shoes. I


## Test on varied prompts

The following `test_decoding` function applies a decoding method, with its parameters, to each prompt of the `inputs` list. It displays the generations and returns them in a list; `show` displays them. Let's test greedy decoding with 50 tokens per prompt.

> **Note to the trainer**: GPT-2 and BelGPT-2 learned from unfiltered web text. Sampling can produce a violent, political or offensive text live, even from neutral prompts like these. Run the demo before the session and read the generations, or comment on the stored outputs rather than rerunning the cells in front of the group.

In [11]:
if lang == "fr":
  inputs = [
    "Je suis allé au",
    "Le train pour Lyon",
    "Comment vas-tu",
    "La recette de la tarte aux pommes commence par",
    "Ça va merci, tu devrais",
    "Le chat de la voisine",
    "En 2050, les voitures",
  ]
else:
  inputs = [
    "I went to the",
    "The train to London",
    "How do you",
    "The apple pie recipe starts with",
    "I'm fine thank you, you should",
    "The neighbour's cat",
    "In 2050, cars",
  ]

In [12]:
def show(generations: list[str]) -> None:
  """Display each prompt of `inputs` with its generation."""
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt: {prompt}")
    print(f"Generation: {generation}")
  print("—" * 80)


def test_decoding(function, *args, **kwargs) -> list[str]:
  """Apply the decoding method `function` to each prompt of `inputs`."""
  generations = [
    function(prompt, *args, **kwargs)
    for prompt in tqdm.auto.tqdm(inputs, desc="Prompts", leave=False)
  ]
  show(generations)
  return generations

In [13]:
greedy_generations = test_decoding(greedy, 50)

————————————————————————————————————————————————————————————————————————————————
Prompt: I went to the
Generation: I went to the store and bought a new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to see the new pair of shoes. I was very excited to
————————————————————————————————————————————————————————————————————————————————
Prompt: The train to London
Generation: The train to London was delayed by a number of hours, with the train carrying passengers to the airport.

The train to London was delayed by a number of hours, with the train carrying passengers to the airport.

The train to London was delayed by a
————————————————————————————————————————————————————————————————————————————————
Prompt: How do you
Generation: How do you know if you're a good candidate for a job?

The answer is simple: you're not.

You're not a good candidate for a job because you're not a good candidate for a job.

You're

## Measuring repetition

Greedy generations quickly "loop": they repeat the same groups of words. To compare decoding methods on this point, let's measure it with the **share of distinct 3-grams** of a text: we split the text into words, form all the triples of consecutive words (the 3-grams), and divide the number of different triples by the total number of triples. It is 1 for a text without repetition, and goes down when passages repeat.

The following `distinct_3grams` function computes it: `"a b c a b c"` has 4 triples, 3 of them different, hence 0.75. The next sections reuse it for each method.

In [14]:
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(zip(words, words[1:], words[2:]))
  return len(set(trigrams)) / len(trigrams)


print(distinct_3grams("a b c a b c"))
print(f"Greedy: {statistics.mean(map(distinct_3grams, greedy_generations)):.2f}")

0.75
Greedy: 0.61


Several greedy generations repeat the same sentence, or the same group of words, until the end: this is a very classic weakness of greedy decoding. The share of distinct 3-grams puts a number on it: it is well below 1, where a text written by a human is close to it.

## Measuring coherence

The share of distinct 3-grams measures repetition, not coherence: a text that no longer makes sense doesn't repeat itself either. For coherence, let's ask the model itself. The **mean log-probability** of a text is the mean, over its tokens, of $\log P(\text{Token}_i | \text{Token}_{j<i})$ according to the model: the closer it is to 0, the more likely the model finds the text, and it goes down when the text takes unlikely turns.

It is computed in a single forward pass over the whole text: the logits of position $i$ give the distribution of token $i + 1$, in which we read the log-probability of the token actually there.

The following `mean_log_prob` function computes it, with [`keras.ops.log_softmax`](https://keras.io/api/ops/nn/#logsoftmax-function) then [`keras.ops.take_along_axis`](https://keras.io/api/ops/numpy/#takealongaxis-function), which reads at each position the log-probability of the actual next token. The `measure` function displays both measures averaged over the generations of a method; the next sections use it for each method.

In [15]:
def mean_log_prob(text: str) -> float:
  token_ids = encode(text)
  logits = causal_lm(
    {"token_ids": token_ids, "padding_mask": keras.ops.ones_like(token_ids)}
  )
  # Position i predicts token i + 1: shift by one position
  log_probs = keras.ops.log_softmax(logits[0, :-1], axis=-1)
  next_ids = token_ids[0, 1:]
  chosen = keras.ops.take_along_axis(log_probs, next_ids[:, None], axis=-1)
  return float(keras.ops.mean(chosen))


def measure(name: str, generations: list[str]) -> None:
  """Display both measures, averaged over the generations of a method."""
  repetition = statistics.mean(map(distinct_3grams, generations))
  log_prob = statistics.mean(map(mean_log_prob, generations))
  print(
    f"{name}: {repetition:.2f} distinct 3-grams, mean log-probability {log_prob:.2f}"
  )


measure("Greedy", greedy_generations)

Greedy: 0.61 distinct 3-grams, mean log-probability -1.73


Greedy decoding has a high mean log-probability: it takes the most probable token at each step, and a loop, once started, becomes more and more probable for the model. A high log-probability is therefore not enough to make a good text: the next sections look at both measures together.

## Sampling

The results of greedy decoding loop quickly and are often very generic. The following methods try to fix these flaws.

The first one is to **sample** from the model's probabilities, rather than always choosing the most probable token: a token with a probability of 0.3 is drawn 3 times out of 10.

The `sampling` function takes the form of `greedy`, but draws the next token with [`keras.random.categorical`](https://keras.io/api/random/random_ops/#categorical-function), which applies the softmax to the logits itself. Let's compare its two measures with those of greedy decoding.

In [16]:
@decoding_loop
def sampling(logits):
  return keras.random.categorical(logits, 1)[0, 0]


generations = test_decoding(sampling, 50)
measure("Sampling", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt: I went to the
Generation: I went to the circuit park to see what that was all about and ended up as a passenger.

See the post below.

Here's what I saw, and the video below from McElándown spotlights how it all started on Super Lucky
————————————————————————————————————————————————————————————————————————————————
Prompt: The train to London
Generation: The train to London from London fell on Christmas Day

A skull carving is due to be unveiled

Delivery company Cuddy bought a majority stake at the weekend, and is looking for lucky customers/survivors for going on the Tube or renting in both year
————————————————————————————————————————————————————————————————————————————————
Prompt: How do you
Generation: How do you research shaping words and sounds in the political world? Get some educated help.

The United States has a very diverse population of roughly 50 million people, and it's quite often pe

The generations don't loop anymore: the share of distinct 3-grams is close to 1. On the other hand, they often lose the thread, because an unlikely token is drawn from time to time and the rest of the text has to make do with it: their mean log-probability is clearly lower than that of greedy decoding.

## Temperature

Before the softmax, we can divide the logits by a **temperature** $T$:

- $T < 1$ sharpens the differences: the model chooses the most probable tokens more often (more coherent, less varied);
- $T > 1$ flattens them: rare tokens are drawn more often (more varied, less coherent);
- $T \to 0$ amounts to greedy decoding.

The `temperature_sampling` function divides the logits by the temperature before the draw. Let's compare the generations and their two measures for $T = 0.5$ and $T = 1.5$.

In [17]:
@decoding_loop
def temperature_sampling(logits, temperature: float):
  return keras.random.categorical(logits / temperature, 1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 50, temperature=temperature)
  measure(f"T = {temperature}", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt: I went to the
Generation: I went to the store a few days ago and bought a few things for my dog. I have a lot of them and I am really happy with how they turned out. My dog loves them, but he has a problem with them not being able to sit up and
————————————————————————————————————————————————————————————————————————————————
Prompt: The train to London
Generation: The train to London was on track to begin at 8am.

The train had stopped at the end of the line at the end of the line at the end of the line at the end of the line at the end of the line at the end of the
————————————————————————————————————————————————————————————————————————————————
Prompt: How do you
Generation: How do you know if you're receiving a card?

If you're receiving a card through your card issuer, it's important to make sure your card is in good working order. This means that you're not getting a transaction fee for your car

At $T = 0.5$, the generations are more coherent: their mean log-probability gets closer to that of greedy decoding; the lower $T$, the closer they get to greedy decoding and its loops. At $T = 1.5$, they hardly make sense anymore: rare tokens are drawn at every step, and their mean log-probability collapses. The share of distinct 3-grams, for its part, tells these two settings apart much less: it measures repetition, not coherence. The temperature sets this trade-off between coherence and variety, which the two measures put numbers on.

## Top-k sampling

A variation of sampling is to draw only among the `k` most probable tokens, to avoid generating a very improbable token.

The `k_sampling` function keeps the `k` largest logits with [`keras.ops.top_k`](https://keras.io/api/ops/core/#topk-function), which also gives their indices in the vocabulary. It draws a position among these `k` (`[None, :]` adds the batch dimension back), then returns the index of the matching token. Let's test it for `k = 20`.

In [18]:
@decoding_loop
def k_sampling(logits, k: int):
  values, indices = keras.ops.top_k(logits[0], k=k)
  sampled = keras.random.categorical(values[None, :], 1)[0, 0]
  return indices[sampled]


generations = test_decoding(k_sampling, 50, k=20)
measure("Top-k", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt: I went to the
Generation: I went to the bar and was met with people yelling and yelling and screaming. When I was asked about it, I said something like, it sounds like you know, a lot of people say that if you've got a problem that doesn't involve any alcohol or drugs
————————————————————————————————————————————————————————————————————————————————
Prompt: The train to London
Generation: The train to London Heathrow will leave on Thursday after being delayed until 10am because "significant issues" remain as a result of the train wreck.

The London Underground train will leave in about four hours, but is now due to leave at 11pm on Sunday
————————————————————————————————————————————————————————————————————————————————
Prompt: How do you
Generation: How do you get your hands on my book, How to Get Into Real Life, which is a memoir by a former professional baseball player who helped set up a company fo

Tokens outside the 20 most probable are never drawn anymore: the generations stay varied, but go off the rails less often than with plain sampling, and their mean log-probability goes back up.

## Top-p sampling

Another improvement is to keep only the most probable tokens whose cumulative probability reaches `p` (the *nucleus* of the distribution), and not the next ones. The number of candidate tokens then adapts to the confidence of the model: few candidates when one token dominates, many when the distribution is flat.

The `k_p_sampling` function applies it after a top-k. It computes the probabilities of the `k` kept tokens with [`keras.ops.softmax`](https://keras.io/api/ops/nn/#softmax-function), then their cumulative sums with [`keras.ops.cumsum`](https://keras.io/api/ops/numpy/#cumsum-function), and keeps the tokens up to the one that crosses `p`, included. For example, for the probabilities `[0.5, 0.3, 0.15, 0.05]` and `p = 0.85`, the cumulative sums are `[0.5, 0.8, 0.95, 1.0]`: 2 are below `p`, we keep 3 tokens. Let's test it for `k = 20` and `p = 0.85`.

In [19]:
@decoding_loop
def k_p_sampling(logits, k: int, p: float):
  values, indices = keras.ops.top_k(logits[0], k=k)
  probabilities = keras.ops.softmax(values)
  # Number of tokens needed for the cumulative probability to reach p
  n_kept = int(keras.ops.sum(keras.ops.cumsum(probabilities) < p)) + 1
  sampled = keras.random.categorical(values[None, :n_kept], 1)[0, 0]
  return indices[sampled]


generations = test_decoding(k_p_sampling, 50, k=20, p=0.85)
measure("Top-k and top-p", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt: I went to the
Generation: I went to the doctor on Sunday and had a heart attack. The doctor said my heart is a bit of a mess.

The doctor gave me a heart attack and I needed surgery. He said, "You're a woman, not a man. You should
————————————————————————————————————————————————————————————————————————————————
Prompt: The train to London
Generation: The train to London's Canary Wharf, on the other hand, is a big one: The train to Liverpool's Liverpool station was a two-car train.

But it's worth asking why it took so long.

When we started talking to passengers
————————————————————————————————————————————————————————————————————————————————
Prompt: How do you
Generation: How do you tell your child about the dangers of drinking? Is it a bad idea? Or are there other ways to help them?

You could help them to find out more about the dangers of drinking.

What is the best way to help them
—————————————

When the model is confident, the nucleus keeps only one or two tokens, and the text follows the most natural continuation; when it hesitates, the draw stays open: the share of distinct 3-grams stays close to 1, and the mean log-probability close to that of top-k. This is the most common setting in practice, along with the temperature.

## Generation with `generate`

In practice, we don't write the decoding loop: the [`generate`](https://keras.io/keras_hub/api/models/gpt2/gpt2_causal_lm/#generate-method) method of KerasHub's models implements all these strategies. Like our `forward`, it keeps the attention keys and values in a cache; it also compiles the whole loop with XLA: the first generation takes about thirty seconds of compilation, the next ones are fast. The choice of the next token is set with `causal_lm.compile(sampler=...)`, with the samplers of [`keras_hub.samplers`](https://keras.io/keras_hub/api/samplers/): `GreedySampler`, `RandomSampler`, `TopKSampler`, `TopPSampler`, which all have a `temperature` parameter.

Let's generate about a hundred tokens for each prompt of `inputs`, in a single call of `generate` on the list (`max_length=110` counts the tokens of the prompt): first with top-p sampling restricted to the 30 most probable tokens, with a nucleus `p = 0.95`, then with greedy decoding. The `seed` parameter fixes the sampler's draws: every run gives back the same texts.

In [20]:
samplers = [
  keras_hub.samplers.TopPSampler(p=0.95, k=30, seed=2),
  keras_hub.samplers.GreedySampler(),
]
for sampler in samplers:
  causal_lm.compile(sampler=sampler)
  generations = causal_lm.generate(inputs, max_length=110)
  show(generations)
  measure(type(sampler).__name__, generations)

————————————————————————————————————————————————————————————————————————————————
Prompt: I went to the
Generation: I went to the grocery store on Sundays. It was just a few days after I bought the food. It was all delicious, fresh vegetables and some fresh fruit. It was quite a long drive to the grocery store. I got my order and there was this woman in a large black tie who called to me. "Where's your bagel?" "It's not here, it's from your parents in the village where you live." She said, "What are you talking about?" "Well, this girl from the village. She's
————————————————————————————————————————————————————————————————————————————————
Prompt: The train to London
Generation: The train to London had its usual brisk and fast departure.

The train began in London, and arrived for the capital, at midnight. By the time you arrived you'd already started your journey.

The train came off, and your journey was only over two hours.

In the meantime you did not want your journey to end so abru

Over a hundred tokens, the gap widens: most greedy generations end up looping, and their share of distinct 3-grams drops, whereas top-p sampling stays varied. Greedy decoding still keeps the highest mean log-probability: its loops are very probable for the model. Neither measure is enough on its own. This is why language models are almost always decoded by sampling, with a temperature, a top-k and a top-p tuned for the intended use.

## A longer text

To finish, let's have the model write a longer text from the beginning of a story, with the same top-p sampling and up to 200 tokens. `generate` stops earlier if the model produces the end of text token.

In [21]:
story = (
  "Il était une fois un petit robot qui"
  if lang == "fr"
  else "Once upon a time, a little robot"
)
causal_lm.compile(sampler=keras_hub.samplers.TopPSampler(p=0.95, k=30, seed=2))
print(causal_lm.generate(story, max_length=200))

Once upon a time, a little robot called Diddy Kong fought his way into the world of Pokémon, and his team had to battle against one of his best foes. But now, Diddy Kong seems to be back in action, battling against a mysterious monster called the Beast, to defeat his friend on his final day to conquer the world of Pokémon.

While it was originally planned to be an adventure game where players pick various Pokémon characters based on their personalities, Diddy Kong is now being played by an actual team of artists: The Team Pokémon, a team of artists who've created the game. And this year they've announced Diddy Kong and Team Rocket.

A first look at Pokémon GO

The game, which will be available to players for "preorder" on August 1, will be made available on Android & iOS devices on August 18th. It's not the first time the Game Freak-inspired series has been in the works since Pokémon Go started life


## Key takeaways

- At each step, the model doesn't produce a word but a score for each token of its vocabulary (50,257 for GPT-2 as for BelGPT-2); a token is a word or a piece of a word.
- The text depends on the way of choosing among these scores, the **decoding**: the same model gives very different texts depending on the setting.
- Greedy decoding (always the most probable) is deterministic, generic, and loops quickly.
- Sampling draws according to the probabilities: more variety, but sometimes incoherent tokens. Top-k and top-p cut off the improbable choices, the **temperature** sets the boldness.
- Two measures put numbers on this trade-off: the share of distinct 3-grams (repetition) and the mean log-probability (coherence, as seen by the model). Neither is enough on its own: greedy decoding has the best log-probability, and loops.
- In practice, we call a `generate` method (KerasHub's here, that of `transformers` in PyTorch), which keeps an attention cache, with these settings.